# Step 4 · Modeling (모델 훈련 — Google Colab)

> **이 노트북은 Google Colab 에서 실행합니다.**
> PC Jupyter Lab에서 열었다면 아래 버튼을 눌러 Colab에서 여세요.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brainai-hub/brainai-car-colab/blob/main/Step%204%20-%20Modeling%20(Colab).ipynb)

## ✅ 시작 전 체크리스트

**1단계 — 런타임 설정 (맨 먼저!)**
> 상단 메뉴 → 런타임 → 런타임 유형 변경 → 아래 두 가지를 고르고 → 저장
>
> - 하드웨어 가속기: **T4 GPU**
> - 런타임 버전: **2026.04** (TensorFlow 2.19.0 이 들어 있는 버전)
>
> 💡 런타임 버전을 바꾸면 연결에 **2~3분** 걸릴 수 있어요. 정상이에요.
> ⚠️ 런타임 버전이 **최신(Latest)** 이면 TensorFlow 버전이 달라 PC에서 모델 변환이 안 될 수 있어요.

**2단계 — 준비할 파일 1개**
- `laneD1_Dataset.zip` — Step 3에서 만든 데이터셋 압축 파일
- 모델 코드는 이 노트북 안에 들어 있어요. 따로 올릴 파일이 없어요.

**3단계 — 실행 순서**

| 순서 | Part | 할 일 | 주의 |
|------|------|-------|------|
| 1 | Part 1 | 한글 폰트 설치 | |
| 2 | Part 2 | 버전 확인 | ❌ 가 나오면 1단계 런타임 버전 확인 |
| 3 | Part 3 | 모델 코드 셀 실행 | |
| 4 | Part 4 | 데이터셋 `.zip` 업로드 → 압축 해제 | |
| 5 | Part 5 | 훈련 설정 확인 | |
| 6 | Part 6 | 훈련 실행 (약 10~20분) | |
| 7 | Part 7 | 결과 그래프 확인 | |
| 8 | Part 8 | 모델 다운로드 | 세션 끊기기 전에 반드시! |

## 전체 흐름

```
① T4 GPU + 런타임 버전 2026.04 설정
② 한글 폰트 설치 → 버전 확인
③ 모델 코드 셀 실행
④ 데이터셋 .zip 업로드 → 압축 해제
⑤ 훈련 실행 (약 10~20분)
⑥ laneD1.keras 다운로드
⑦ PC 의 models/laneD1/ 폴더에 복사
```

## Part 1. 한글 폰트 설치

> ⚠️ **Colab 전용** — PC JupyterLab 에서는 이 셀을 실행하지 마세요.

그래프에 한글이 깨지지 않도록 폰트를 설치합니다.

> 💡 TensorFlow 는 설치하지 않아요. 런타임 버전 **2026.04** 에 TensorFlow 2.19.0 이 이미 들어 있어요.

In [ ]:
# Colab 전용 — PC JupyterLab 에서는 실행하지 마세요
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1   # 그래프 한글 폰트
print("✅ 한글 폰트 설치 완료")

## Part 2. 버전 확인

TensorFlow 2.19.0 / GPU: True 가 출력되면 정상입니다.

❌ 가 나오면 **런타임 → 런타임 유형 변경** 에서 런타임 버전 **2026.04** · **T4 GPU** 를 선택한 뒤 이 셀부터 다시 실행하세요.

In [ ]:
import tensorflow as tf

REQUIRED_TF_VERSION = "2.19.0"   # PC 의 Step 5 (OpenVINO 변환) 와 같은 버전이어야 해요
has_gpu = len(tf.config.list_physical_devices('GPU')) > 0

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {tf.keras.__version__}")
print(f"GPU 사용   : {has_gpu}")

if tf.__version__ != REQUIRED_TF_VERSION:
    print(f"\n❌ TensorFlow {tf.__version__} 입니다. ({REQUIRED_TF_VERSION} 이 필요해요)")
    print("👉 상단 메뉴: 런타임 → 런타임 유형 변경 → 런타임 버전 '2026.04' 선택 → 저장")
    print("   연결된 뒤 Part 1 부터 다시 실행하세요.")
elif not has_gpu:
    print("\n❌ GPU 가 연결되지 않았어요.")
    print("👉 상단 메뉴: 런타임 → 런타임 유형 변경 → 하드웨어 가속기 'T4 GPU' 선택 → 저장")
else:
    print("\n✅ 버전 확인 완료 — 다음 Part 로 진행하세요")

## Part 3. 모델 코드 불러오기

아래 셀을 실행하면 훈련에 필요한 모델 코드가 준비됩니다.

> 💡 코드가 길지만 **실행만 하면 돼요.** 내용은 수정하지 마세요.

In [ ]:
# ── 모델 코드 (course/utils/modeling.py 와 동일) ─────────────────────────
# 이 셀을 실행하면 TrainingConfig / CNNTrainer 가 메모리에 로드됩니다.
# ★ 이미지 전처리는 자동차 추론(lane_detection.py)과 반드시 같아야 해서, 내용을 바꾸지 마세요.

import json
import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras import layers, models, utils
from tensorflow import keras


# ─── TrainingConfig ───────────────────────────────────────────────────────────

@dataclass
class TrainingConfig:
    """
    CNN 훈련 설정.

    필수:
        dataset_path: create_dataset.py 로 만든 데이터셋 폴더 경로.
        model_name:   저장할 모델 이름. models/{model_name}/{model_name}.keras 로 저장됨.

    선택:
        epochs:       최대 훈련 에폭 (EarlyStopping 으로 조기 종료 가능).
        batch_size:   배치 크기.
        img_height, img_width: 입력 이미지 크기. lane_detection.py 와 반드시 동일해야 함.
        learning_rate: Adam 옵티마이저 학습률.
        use_augmentation: 데이터 증강 사용 여부.
        augmentation_options: 증강 항목별 ON/OFF.
    """
    dataset_path: str
    model_name:   str
    epochs:       int   = 50
    batch_size:   int   = 32
    img_height:   int   = 224
    img_width:    int   = 224
    learning_rate: float = 0.001
    use_augmentation: bool = False
    augmentation_options: Dict[str, bool] = field(default_factory=lambda: {
        "COLOR_JITTER":     False,
        "GAUSSIAN_NOISE":   False,
    })
    # Colab 전용(선택): 지정 시 훈련 후 .keras 를 이 폴더에도 복사 (런타임 끊겨도 보존)
    drive_save_dir: str = ""

    def __post_init__(self) -> None:
        if not self.dataset_path:
            raise ValueError("dataset_path cannot be empty.")
        if not self.model_name:
            raise ValueError("model_name cannot be empty.")
        if self.epochs <= 0 or self.batch_size <= 0:
            raise ValueError("epochs and batch_size must be positive.")

    def print_summary(self) -> None:
        print("=" * 55)
        print("Training Configuration")
        print("=" * 55)
        print(f"  Dataset:       {self.dataset_path}")
        print(f"  Model name:    {self.model_name}")
        print(f"  Input size:    {self.img_width}×{self.img_height}")
        print(f"  Epochs:        {self.epochs}")
        print(f"  Batch size:    {self.batch_size}")
        print(f"  Learning rate: {self.learning_rate}")
        print(f"  Augmentation:  {'ON' if self.use_augmentation else 'OFF'}")
        if self.use_augmentation:
            for k, v in self.augmentation_options.items():
                print(f"    - {k}: {'yes' if v else 'no'}")
        print(f"\n  steering range: -1.0 ~ +1.0 (raw, no re-normalization)")
        print("=" * 55)


# ─── DataLoader ───────────────────────────────────────────────────────────────

class DataLoader:
    """
    이미지 경로와 조향값을 로드합니다.

    steering 값 기준: -1.0~+1.0 (data_collector.py 저장 기준과 동일).
    별도 정규화 없이 그대로 사용합니다.
    """

    @staticmethod
    def load(images_path: str, annotations_path: str) -> Tuple[List[str], np.ndarray]:
        """
        이미지 경로 목록과 조향값 배열을 반환합니다.

        Returns:
            (image_paths, annotations)
            annotations shape: (N, 1), dtype: float32, range: -1.0~+1.0
        """
        if not os.path.exists(images_path) or not os.path.exists(annotations_path):
            print(f"[WARN] Path not found: {images_path} or {annotations_path}")
            return [], np.array([])

        image_paths: List[str] = []
        annotations: List[List[float]] = []

        for ann_file in sorted(os.listdir(annotations_path)):
            if not ann_file.endswith(".json"):
                continue

            ann_path = os.path.join(annotations_path, ann_file)
            base = os.path.splitext(ann_file)[0]

            # 대응하는 이미지 파일 탐색
            img_path = None
            for ext in (".jpg", ".jpeg", ".png"):
                candidate = os.path.join(images_path, base + ext)
                if os.path.exists(candidate):
                    img_path = candidate
                    break

            if img_path is None:
                continue

            # JSON 로드 및 범위 검증
            try:
                with open(ann_path) as f:
                    data = json.load(f)
                steering = float(data["steering"])
                if not (-1.0 <= steering <= 1.0):
                    print(f"[WARN] steering out of range ({steering:.3f}): {ann_file}")
                    continue
            except (KeyError, ValueError, json.JSONDecodeError) as e:
                print(f"[WARN] Could not load annotation {ann_file}: {e}")
                continue

            image_paths.append(img_path)
            annotations.append([steering])

        return image_paths, np.array(annotations, dtype=np.float32)


# ─── ImageAugmentation ────────────────────────────────────────────────────────

class ImageAugmentation:
    """이미지 증강 유틸리티."""

    @staticmethod
    def color_jitter(img: tf.Tensor, strength: float = 0.2) -> tf.Tensor:
        """밝기·대비·채도를 랜덤하게 변환합니다."""
        if tf.random.uniform([]) > 0.5:
            img = tf.image.adjust_brightness(img, tf.random.uniform([], -strength, strength))
        if tf.random.uniform([]) > 0.5:
            img = tf.image.adjust_contrast(img, tf.random.uniform([], 1 - strength, 1 + strength))
        if tf.random.uniform([]) > 0.5:
            img = tf.image.adjust_saturation(img, tf.random.uniform([], 1 - strength, 1 + strength))
        return img

    @staticmethod
    def gaussian_noise(img: tf.Tensor, stddev: float = 0.05) -> tf.Tensor:
        """가우시안 노이즈를 추가합니다."""
        if tf.random.uniform([]) > 0.5:
            noise = tf.random.normal(shape=tf.shape(img), stddev=stddev)
            img = tf.clip_by_value(img + noise, 0.0, 1.0)
        return img


# ─── CNNTrainer ───────────────────────────────────────────────────────────────

class CNNTrainer:
    """
    MobileNetV3Large 기반 CNN 회귀 모델 훈련 클래스.

    모델 구조:
      MobileNetV3Large (pretrained, frozen)
        → GlobalAveragePooling2D
        → Dense(64, relu)
        → Dropout(0.2)
        → Dense(32, relu)
        → Dense(1)           ← 조향값 출력 (-1.0~+1.0)
    """

    def __init__(self, config: TrainingConfig):
        self.config = config
        self.model: keras.Model | None = None
        self.history: keras.callbacks.History | None = None

        self._train_images      = os.path.join(config.dataset_path, "train", "images")
        self._train_annotations = os.path.join(config.dataset_path, "train", "annotations")
        self._val_images        = os.path.join(config.dataset_path, "validation", "images")
        self._val_annotations   = os.path.join(config.dataset_path, "validation", "annotations")

    # ── 훈련 진입점 ───────────────────────────────────────────────────────────

    def train(self) -> Tuple[keras.Model, keras.callbacks.History]:
        """
        모델을 훈련하고 (model, history) 를 반환합니다.
        최적 모델은 models/{model_name}/{model_name}.keras 에 자동 저장됩니다.
        """
        self.config.print_summary()

        # 데이터 로드
        train_ds, val_ds, n_train, n_val = self._load_datasets()
        print(f"\n  Train: {n_train} samples | Validation: {n_val} samples")

        # 모델 빌드
        self.model = self._build_model()
        print(self.model.summary())

        # 훈련
        print(f"\n{'='*55}\nStarting training...\n{'='*55}")
        import time
        _start = time.time()
        self.history = self.model.fit(
            train_ds,
            epochs=self.config.epochs,
            validation_data=val_ds,
            callbacks=self._callbacks(val_ds),
        )

        # 총 훈련 시간 출력
        _elapsed = time.time() - _start
        _m, _s = divmod(int(_elapsed), 60)
        _h, _m = divmod(_m, 60)
        if _h > 0:
            _time_str = f"{_h}시간 {_m}분 {_s}초"
        elif _m > 0:
            _time_str = f"{_m}분 {_s}초"
        else:
            _time_str = f"{_s}초"
        print(f"\n⏱️  총 훈련 시간: {_time_str} ({_elapsed:.1f}초)")

        # 결과 출력
        self._print_results()
        self._plot_history()

        # Google Drive 자동 저장 (Colab에서 drive_save_dir 지정 시) — 런타임 끊겨도 보존
        if self.config.drive_save_dir:
            try:
                import shutil
                os.makedirs(self.config.drive_save_dir, exist_ok=True)
                src = os.path.join("models", self.config.model_name, f"{self.config.model_name}.keras")
                dst = os.path.join(self.config.drive_save_dir, f"{self.config.model_name}.keras")
                if os.path.exists(src):
                    shutil.copy2(src, dst)
                    print(f"📁 Google Drive에도 저장됨: {dst}")
                else:
                    print(f"[WARN] 저장된 모델을 찾지 못해 Drive 복사를 건너뜀: {src}")
            except Exception as e:
                print(f"[WARN] Drive 저장 실패: {e}")

        return self.model, self.history

    # ── 이미지 전처리 ─────────────────────────────────────────────────────────

    def preprocess_image(self, img_path_tensor: Any, augment: bool = False) -> Tuple[tf.Tensor, None]:
        """
        이미지를 모델 입력 형식으로 변환합니다.

        ★ 이 전처리는 lane_detection.py 의 _preprocess() 와 반드시 동일해야 합니다.
          1. 리사이즈 (config.img_width × config.img_height)
          2. [0, 1] 정규화
          3. 상단 절반 0으로 마스킹
          4. [-1, 1] 스케일 변환
        """
        # 텐서 → 파일 경로 문자열 변환
        if isinstance(img_path_tensor, tf.Tensor):
            path = img_path_tensor.numpy().decode("utf-8")
        else:
            path = str(img_path_tensor)

        img = utils.load_img(path, target_size=(self.config.img_height, self.config.img_width))
        img_arr = utils.img_to_array(img) / 255.0

        # 데이터 증강 (훈련 시만)
        if augment and self.config.use_augmentation:
            opts = self.config.augmentation_options
            if opts.get("COLOR_JITTER"):
                img_arr = ImageAugmentation.color_jitter(img_arr)
            if opts.get("GAUSSIAN_NOISE"):
                img_arr = ImageAugmentation.gaussian_noise(img_arr)
            # ※ HORIZONTAL_FLIP 은 조향값도 반전해야 하므로 현재 미지원
            #   (라벨도 같이 뒤집는 로직은 추후 추가)

        # 상단 절반 마스킹 (lane_detection.py 와 동일)
        half_h = self.config.img_height // 2
        zeros = tf.zeros_like(img_arr[:half_h, :, :])
        img_arr = tf.concat([zeros, img_arr[half_h:, :, :]], axis=0)

        # [-1, 1] 스케일 (lane_detection.py 와 동일)
        img_arr = img_arr * 2.0 - 1.0

        return img_arr, None

    # ── 내부 구현 ─────────────────────────────────────────────────────────────

    def _build_model(self) -> keras.Model:
        base = keras.applications.MobileNetV3Large(
            input_shape=(self.config.img_height, self.config.img_width, 3),
            include_top=False,
            weights="imagenet",
            minimalistic=False,
            alpha=0.75,
        )
        base.trainable = False

        model = models.Sequential([
            base,
            layers.GlobalAveragePooling2D(),
            layers.Dense(64, activation="relu"),
            layers.Dropout(0.2),
            layers.Dense(32, activation="relu"),
            layers.Dense(1),  # 출력: -1.0~+1.0 (activation 없음)
        ])
        model.compile(
            optimizer=keras.optimizers.Adam(self.config.learning_rate),
            loss="mse",
            metrics=["mae"],
        )
        return model

    def _create_tf_dataset(
        self,
        image_paths: List[str],
        annotations: np.ndarray,
        augment: bool = False,
    ) -> tf.data.Dataset:
        path_ds = tf.data.Dataset.from_tensor_slices(image_paths)
        ann_ds  = tf.data.Dataset.from_tensor_slices(annotations)
        dataset = tf.data.Dataset.zip((path_ds, ann_ds))

        h, w = self.config.img_height, self.config.img_width

        def _map(path, ann):
            img = tf.py_function(
                func=lambda p: self.preprocess_image(p, augment)[0],
                inp=[path],
                Tout=tf.float32,
            )
            img.set_shape([h, w, 3])
            return img, ann

        return (
            dataset
            .map(_map, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(self.config.batch_size)
            .prefetch(tf.data.AUTOTUNE)
        )

    def _load_datasets(self):
        print("Loading train dataset...")
        train_imgs, train_anns = DataLoader.load(self._train_images, self._train_annotations)
        print("Loading validation dataset...")
        val_imgs,   val_anns   = DataLoader.load(self._val_images,   self._val_annotations)

        if not train_imgs:
            raise ValueError("No training data found. Check dataset_path.")

        train_ds = self._create_tf_dataset(train_imgs, train_anns, augment=self.config.use_augmentation)
        val_ds   = self._create_tf_dataset(val_imgs,   val_anns,   augment=False) if val_imgs else None

        return train_ds, val_ds, len(train_imgs), len(val_imgs)

    def _callbacks(self, val_ds) -> list:
        # val_mae: 실제 조향 정확도와 직결 → val_loss(MSE) 보다 직관적
        monitor = "val_mae" if val_ds else "mae"
        model_dir  = os.path.join("models", self.config.model_name)
        os.makedirs(model_dir, exist_ok=True)
        save_path  = os.path.join(model_dir, f"{self.config.model_name}.keras")

        return [
            keras.callbacks.ModelCheckpoint(
                save_path, monitor=monitor, save_best_only=True, verbose=1),
            keras.callbacks.EarlyStopping(
                monitor=monitor,
                patience=10,      # 10 에포크 동안 개선 없으면 종료 (기존 5)
                verbose=1,
                restore_best_weights=True),   # 종료 시 최적 가중치 복원
            keras.callbacks.ReduceLROnPlateau(
                monitor=monitor,
                factor=0.5,
                patience=5,       # 5 에포크 후 학습률 절반 (기존 3)
                min_lr=1e-6,      # 학습률 최솟값 제한
                verbose=1),
        ]

    def _print_results(self) -> None:
        if self.history is None:
            return
        h = self.history.history
        print(f"\n{'='*55}")
        print("Training complete.")
        print(f"  Final train  loss: {h['loss'][-1]:.4f}  MAE: {h['mae'][-1]:.4f}")
        if "val_loss" in h:
            print(f"  Final val    loss: {h['val_loss'][-1]:.4f}  MAE: {h['val_mae'][-1]:.4f}")
        print(f"{'='*55}")

    def _plot_history(self) -> None:
        if self.history is None:
            return

        # 한글 폰트 설정 (Colab: NanumGothic, PC: 시스템 폰트 자동 탐색)
        import glob
        import matplotlib.font_manager as fm
        # Colab: fonts-nanum 설치 시 런타임에 등록 (폰트 캐시에 없을 수 있음)
        for _path in glob.glob("/usr/share/fonts/truetype/nanum/*.ttf"):
            fm.fontManager.addfont(_path)
        _korean_fonts = ["NanumGothic", "NanumBarunGothic", "Malgun Gothic", "AppleGothic", "DejaVu Sans"]
        _available = {f.name for f in fm.fontManager.ttflist}
        for _font in _korean_fonts:
            if _font in _available:
                plt.rcParams["font.family"] = _font
                break
        plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지

        h = self.history.history
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        for ax, metric, title in zip(axes, ["loss", "mae"], ["Loss (낮을수록 좋음)", "MAE (낮을수록 좋음)"]):
            ax.plot(h[metric], label="훈련")
            if f"val_{metric}" in h:
                ax.plot(h[f"val_{metric}"], label="검증")
            ax.set_title(title)
            ax.set_xlabel("Epoch")
            ax.legend()
            ax.grid(True)

        plt.tight_layout()
        plot_path = f"training_history_{self.config.model_name}.png"
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"[Trainer] History plot saved: {plot_path}")

print("✅ 모델 코드 로드 완료 (TrainingConfig, CNNTrainer)")

## Part 4. 데이터셋 업로드 및 압축 해제

Step 3 에서 만든 `laneD1_Dataset.zip` 을 업로드합니다.

업로드 방법:
1. Colab 왼쪽 📁 아이콘 클릭
2. ⬆️ 업로드 버튼 클릭
3. `laneD1_Dataset.zip` 선택
4. 업로드 완료 후 아래 셀 실행

> 💡 `replace ... ? [y]es, [n]o, [A]ll ...` 메시지 → **A + Enter** 입력

In [ ]:
import os

# ✏️ 여기만 수정하세요 ──────────────────────────────────
DATASET_NAME = "laneD1_Dataset"   # zip 파일 이름 (확장자 제외)
MODEL_NAME   = "laneD1"           # 저장할 모델 이름
# ────────────────────────────────────────────────────────

!mkdir -p /content/datasets
!unzip -q {DATASET_NAME}.zip -d /content/datasets/

DATASET_PATH = f"/content/datasets/{DATASET_NAME}"

train_dir = os.path.join(DATASET_PATH, "train", "images")
val_dir   = os.path.join(DATASET_PATH, "validation", "images")

print(f"데이터셋 경로 : {DATASET_PATH}")
print(f"train/images  : {'✅' if os.path.exists(train_dir) else '❌ 경로를 확인하세요'}")
print(f"validation    : {'✅' if os.path.exists(val_dir)   else '❌ 경로를 확인하세요'}")

if os.path.exists(train_dir):
    n_train = len([f for f in os.listdir(train_dir) if f.endswith(('.jpg','.jpeg','.png'))])
    print(f"\n훈련 이미지  : {n_train} 장")
if os.path.exists(val_dir):
    n_val = len([f for f in os.listdir(val_dir) if f.endswith(('.jpg','.jpeg','.png'))])
    print(f"검증 이미지  : {n_val} 장")

## Part 5. 훈련 설정

| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| `epochs` | 30 | 최대 반복 횟수 (EarlyStopping 으로 조기 종료 가능) |
| `batch_size` | 16 | Colab T4 GPU 메모리 기준 |
| `learning_rate` | 0.001 | 학습 속도 |

In [ ]:
# Google Drive 마운트 — 훈련 결과를 Drive에 보존 (런타임이 끊겨도 안전)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/BrainAI/models'
import os
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Drive 저장 경로: {DRIVE_SAVE_DIR}")

In [ ]:
config = TrainingConfig(
    dataset_path=DATASET_PATH,
    model_name=MODEL_NAME,
    epochs=30,
    batch_size=16,        # Colab T4 GPU 메모리 기준 (PC 는 32)
    learning_rate=0.001,
    use_augmentation=False,
    drive_save_dir=DRIVE_SAVE_DIR,   # 훈련 후 Drive에도 자동 복사
)

config.print_summary()

## Part 6. 훈련 실행

> ⏱️ 예상 시간: T4 GPU 기준 약 10~20분

In [ ]:
trainer = CNNTrainer(config)
model, history = trainer.train()

## Part 7. 결과 확인

- **Loss / MAE 그래프** : 검증 loss 가 계속 내려가면 ✅, 올라가면 과적합 신호
- **MAE 기준** : 0.20 미만 우수 / 0.30 미만 양호 / 0.40 미만 보통 / 0.40 이상 데이터 추가 수집 권장

In [ ]:
from IPython.display import Image, display

# 훈련 그래프 보기 (훈련 중 자동 저장된 이미지)
plot_path = f"training_history_{MODEL_NAME}.png"
if os.path.exists(plot_path):
    display(Image(plot_path))

final_mae = min(history.history['val_mae'])
print(f"\n최종 검증 MAE : {final_mae:.4f}")
if final_mae < 0.20:
    print("✅ 우수 — 자율주행 적합")
elif final_mae < 0.30:
    print("✅ 양호 — 대부분 상황에서 작동")
elif final_mae < 0.40:
    print("⚠️ 보통 — 데이터 추가 권장")
else:
    print("❌ 개선 필요 — 데이터 재수집 필요")

## Part 8. 모델 다운로드

> ⚠️ **Colab은 세션이 끊기면 모든 파일이 사라집니다. 훈련 직후 반드시 다운로드하세요!**

다운로드 후:
1. `laneD1.keras` 파일을 PC의 `models\laneD1\` 폴더에 복사
2. Step 5 (OpenVINO 최적화) 노트북 실행

In [ ]:
from google.colab import files
import os

model_path = os.path.join('models', MODEL_NAME, f'{MODEL_NAME}.keras')

if os.path.exists(model_path):
    files.download(model_path)
    print(f"✅ 다운로드 시작: {model_path}")
    print(f"\n다음: 다운로드된 {MODEL_NAME}.keras 를 PC 의 models/{MODEL_NAME}/ 폴더에 넣으세요")
else:
    print("❌ 모델 파일이 없습니다. 훈련이 완료됐는지 확인하세요.")

print("\n💡 런타임이 끊겨도 모델은 Google Drive에 저장돼 있어요 — 구글 드라이브 → BrainAI → models 폴더에서 확인하세요.")

## 다음 ▶️ Step 5 · Model Optimization

다운로드된 `laneD1.keras` 를 PC 의 `models/laneD1/` 폴더에 복사한 뒤
Step 5 (OpenVINO 최적화) 노트북을 실행하세요.

```
models/
└── laneD1/
    └── laneD1.keras  ← 여기에 복사
```